# Fine-tuning Gemma 4 E2B for ABSA

In this notebook we train a LoRA adapter on the SemEval restaurant training split, then evaluate it.

The training prompt is the **same** zero-shot prompt the main notebook uses, so the adapter learns to answer exactly the input it will be served at inference time.

## 1. Runtime check and Drive

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU"
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
PROJECT = Path("/content/drive/MyDrive/absa")
DATA_DIR = PROJECT / "data"
OUT_DIR  = PROJECT / "finetune" / "gemma4_e2b"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("train.json present:", (DATA_DIR / "train.json").exists())

GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train.json present: True


## 2. Install Unsloth

In [ ]:
%%capture
!pip install -U unsloth unsloth_zoo
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
import unsloth, transformers, trl
print("unsloth", unsloth.__version__, "| transformers", transformers.__version__, "| trl", trl.__version__)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth 2026.5.10 | transformers 5.5.0 | trl 0.24.0


## 3. Build instruction data from the split

Each example is `(prompt, completion)`: the prompt is the notebook's zero-shot prompt for the
sentence, the completion is the gold aspect/sentiment list as compact JSON. Sentences with no aspect
term map to `[]`, which teaches the model to give empty array rather than hallucinate aspects.

`SYSTEM_INSTRUCTION` / `build_prompt` are copied from the main notebook so the formats can't
drift apart.

In [ ]:
import json
SENTIMENTS = ["positive", "negative", "neutral", "conflict"]

SYSTEM_INSTRUCTION = (
    "You are an aspect-based sentiment analysis system for customer reviews.\n"
    "Given one sentence, extract every aspect term that is explicitly mentioned and its sentiment.\n"
    "An aspect term is a word or phrase copied verbatim from the sentence that names something being "
    "evaluated.\n"
    "sentiment is exactly one of: positive, negative, neutral, conflict "
    "(conflict = both positive and negative).\n"
    "If the sentence contains no aspect term, return an empty array.\n"
    "Return ONLY a JSON array and nothing else. "
    'Each element must be {"aspect": "<term>", "sentiment": "<sentiment>"}.'
)

def build_prompt(text, examples=None):
    parts = [SYSTEM_INSTRUCTION, ""]
    if examples:
        for ex in examples:
            parts.append(f'Sentence: "{ex["text"]}"')
            parts.append("Output: " + json.dumps(
                [{"aspect": p["aspect"], "sentiment": p["sentiment"]} for p in ex["aspects"]],
                ensure_ascii=False))
            parts.append("")
    parts.append(f'Sentence: "{text}"')
    parts.append("Output:")
    return "\n".join(parts)

train = json.loads((DATA_DIR / "train.json").read_text(encoding="utf-8"))
print("train sentences:", len(train))

def to_completion(s):
    return json.dumps([{"aspect": p["aspect"], "sentiment": p["sentiment"]} for p in s["aspects"]],
                      ensure_ascii=False)

rows = [{"prompt": build_prompt(s["text"]), "completion": to_completion(s)} for s in train]
print("example:\n", rows[0]["prompt"], "\n-> ", rows[0]["completion"])

train sentences: 1616
example:
 You are an aspect-based sentiment analysis system for customer reviews.
Given one sentence, extract every aspect term that is explicitly mentioned and its sentiment.
An aspect term is a word or phrase copied verbatim from the sentence that names something being evaluated.
sentiment is exactly one of: positive, negative, neutral, conflict (conflict = both positive and negative).
If the sentence contains no aspect term, return an empty array.
Return ONLY a JSON array and nothing else. Each element must be {"aspect": "<term>", "sentiment": "<sentiment>"}.

Sentence: "When you want a piece of beef, head on over."
Output: 
->  [{"aspect": "beef", "sentiment": "positive"}]


## 4. Load Gemma 4 E2B (4-bit) and attach LoRA

QLoRA: the base model loads in 4-bit, only the LoRA adapters train.

In [ ]:
from unsloth import FastModel
import torch

MAX_SEQ = 1024  # ABSA inputs are short; small value saves VRAM and speeds training

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    max_seq_length = MAX_SEQ,
    dtype = None,
    load_in_4bit = True, # QLoRA
    full_finetuning = False,
)

model = FastModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

==((====))==  Unsloth 2026.5.10: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/301k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/16.8k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

## 5. Format and tokenize

In [ ]:
from datasets import Dataset

EOS = tokenizer.eos_token
def to_text(r):
    return {"text": r["prompt"] + " " + r["completion"] + EOS}

ds = Dataset.from_list(rows).map(to_text)
ds = ds.train_test_split(test_size=0.05, seed=42)
print(ds)
print(ds["train"][0]["text"])

Map:   0%|          | 0/1616 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion', 'text'],
        num_rows: 1535
    })
    test: Dataset({
        features: ['prompt', 'completion', 'text'],
        num_rows: 81
    })
})
You are an aspect-based sentiment analysis system for customer reviews.
Given one sentence, extract every aspect term that is explicitly mentioned and its sentiment.
An aspect term is a word or phrase copied verbatim from the sentence that names something being evaluated.
sentiment is exactly one of: positive, negative, neutral, conflict (conflict = both positive and negative).
If the sentence contains no aspect term, return an empty array.
Return ONLY a JSON array and nothing else. Each element must be {"aspect": "<term>", "sentiment": "<sentiment>"}.

Sentence: "Nicky the Nose at the bar is a treat."
Output: [{"aspect": "bar", "sentiment": "neutral"}]<eos>


## 6. Train (3 epochs)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds["train"],
    eval_dataset = ds["test"],
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,     # effective batch 8
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 10,
        eval_strategy = "epoch",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = str(OUT_DIR / "checkpoints"),
        report_to = "none",
    ),
)

stats = trainer.train()
print(stats)

Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=4):   0%|          | 0/1535 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=4):   0%|          | 0/81 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,535 | Num Epochs = 3 | Total steps = 576
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 31,039,488 of 5,154,217,504 (0.60% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Epoch,Training Loss,Validation Loss
1,0.190784,0.788230
2,0.069666,0.702740
3,0.045449,0.690689


Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/absa/finetune/gemma4_e2b/checkpoints/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/absa/finetune/gemma4_e2b/checkpoints/checkpoint-576/tokenizer_config.json.


TrainOutput(global_step=576, training_loss=0.18729565202051568, metrics={'train_runtime': 2280.794, 'train_samples_per_second': 2.019, 'train_steps_per_second': 0.253, 'total_flos': 1.15621967982048e+16, 'train_loss': 0.18729565202051568, 'epoch': 3.0})


## 7. Evaluate on test set

GGUF conversion requires more RAM than the free Colab T4 runtime provides for this model size.
The fine-tuned model is therefore evaluated here directly via transformers (4-bit NF4 quantization),
using the same prompt, parsing logic, and output format as all other models in the main notebook.
The resulting JSONL file is saved to Drive and downloaded. After that it is droped into the main notebook's
`predictions/` folder and is picked up by the evaluation cell without any changes.

NF4 bitsandbytes here and Q4_K_M GGUF both are 4-bit, the practical difference is small.

In [ ]:
import json, re, time
from pathlib import Path

SENTIMENTS = ["positive", "negative", "neutral", "conflict"]
NULL_TOKENS = {"", "null", "nan", "none"}

SYSTEM_INSTRUCTION = (
    "You are an aspect-based sentiment analysis system for customer reviews.\n"
    "Given one sentence, extract every aspect term that is explicitly mentioned and its sentiment.\n"
    "An aspect term is a word or phrase copied verbatim from the sentence that names something being "
    "evaluated.\n"
    "sentiment is exactly one of: positive, negative, neutral, conflict "
    "(conflict = both positive and negative).\n"
    "If the sentence contains no aspect term, return an empty array.\n"
    "Return ONLY a JSON array and nothing else. "
    'Each element must be {"aspect": "<term>", "sentiment": "<sentiment>"}.'
)

def build_prompt(text, examples=None):
    parts = [SYSTEM_INSTRUCTION, ""]
    if examples:
        for ex in examples:
            parts.append(f'Sentence: "{ex["text"]}"')
            parts.append("Output: " + json.dumps(
                [{"aspect": p["aspect"], "sentiment": p["sentiment"]} for p in ex["aspects"]],
                ensure_ascii=False))
            parts.append("")
    parts.append(f'Sentence: "{text}"')
    parts.append("Output:")
    return "\n".join(parts)

def _strip_think(text):
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)

def _extract_array(text):
    s, e = text.find("["), text.rfind("]")
    return text[s:e + 1] if (s != -1 and e > s) else None

def _normalize_pairs(data):
    if isinstance(data, dict):
        for k in ("aspects", "results", "data", "items"):
            if isinstance(data.get(k), list):
                data = data[k]; break
        else:
            data = [data] if ("aspect" in data or "term" in data) else None
    if not isinstance(data, list):
        return None
    out = []
    for item in data:
        if not isinstance(item, dict): continue
        a = item.get("aspect", item.get("term"))
        s = item.get("sentiment", item.get("polarity"))
        if a is None or s is None: continue
        out.append({"aspect": str(a).strip(), "sentiment": str(s).strip().lower()})
    return out

def parse_prediction(raw):
    # returns (pairs, ok); ok=False -> output was not parseable as the expected JSON
    text = re.sub(r"```(?:json)?", "", _strip_think(str(raw))).strip()
    for cand in (text, _extract_array(text)):
        if not cand: continue
        try:
            data = json.loads(cand)
        except Exception:
            continue
        pairs = _normalize_pairs(data)
        if pairs is not None:
            return pairs, True
    return [], False

test = json.loads((DATA_DIR / "test.json").read_text(encoding="utf-8"))
print(f"test sentences: {len(test)}")

test sentences: 405


In [ ]:
from unsloth import FastModel
import gc

# free training RAM before loading for inference
for name in ["trainer"]:
    if name in globals():
        del globals()[name]
gc.collect(); torch.cuda.empty_cache()

# load fine-tuned model from checkpoint
CKPT = OUT_DIR / "checkpoints" / "checkpoint-576"
model, tokenizer = FastModel.from_pretrained(
    model_name = str(CKPT),
    max_seq_length = 1024,
    dtype = None,
    load_in_4bit = True,
)
FastModel.for_inference(model)
print("model ready for inference")

==((====))==  Unsloth 2026.5.10: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/301k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/16.8k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model ready for inference


In [ ]:
PRED_PATH = PROJECT / "predictions" / "gemma4_e2b_finetuned__finetuned.jsonl"
PRED_PATH.parent.mkdir(parents=True, exist_ok=True)

def load_done(path):
    if not path.exists(): return set()
    ids = set()
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip(): ids.add(json.loads(line)["id"])
    return ids

done = load_done(PRED_PATH)
todo = [s for s in test if s["id"] not in done]
print(f"{len(todo)} to do ({len(done)} cached)")

with PRED_PATH.open("a", encoding="utf-8") as f:
    for i, s in enumerate(todo, 1):
        prompt = build_prompt(s["text"])

        # Gemma 4 is multimodal: content must be a list of typed parts
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_tensors="pt",
            return_dict=True,
        ).to("cuda")

        t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
        dt = time.perf_counter() - t0

        raw = tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        pairs, ok = parse_prediction(raw)

        rec = {"id": s["id"], "text": s["text"], "gold": s["aspects"],
               "pred": pairs, "ok": ok, "time": dt, "mem_mb": None}
        f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush()

        if i % 25 == 0 or i == len(todo):
            print(f"  {i}/{len(todo)}, last raw: {raw[:60]!r}")

print("done ->", PRED_PATH)

405 to do (0 cached)
  25/405, last raw: '[{"aspect": "crust", "sentiment": "negative"}, {"aspect": "i'
  50/405, last raw: '[\n  {\n    "aspect": "food",\n    "sentiment": "positive"\n  },'
  75/405, last raw: '[{"aspect": "kitchen", "sentiment": "negative"}]'
  100/405, last raw: '[{"aspect": "mushroom pizza", "sentiment": "positive"}]'
  125/405, last raw: '[{"aspect": "Japanese cuisine", "sentiment": "negative"}]'
  150/405, last raw: '[\n  {\n    "aspect": "Service",\n    "sentiment": "negative"\n '
  175/405, last raw: '[{"aspect": "pizza", "sentiment": "positive"}]'
  200/405, last raw: '[{"aspect": "dinosaur rolls", "sentiment": "neutral"}, {"asp'
  225/405, last raw: '[{"aspect": "price", "sentiment": "positive"}, {"aspect": "s'
  250/405, last raw: '[{"aspect": "decor", "sentiment": "negative"}]'
  275/405, last raw: '[{"aspect": "waitress", "sentiment": "positive"}, {"aspect":'
  300/405, last raw: '[{"aspect": "Service", "sentiment": "positive"}]'
  325/405, last raw: '[{"a

In [ ]:
records = [json.loads(l) for l in PRED_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
ok = [r for r in records if r["ok"]]
print(f"total: {len(records)}, parsed: {len(ok)}, compliance: {len(ok)/len(records):.1%}")
print("sample output:", records[0]["pred"])

from google.colab import files
files.download(str(PRED_PATH))

total: 405, parsed: 402, compliance: 99.3%
sample output: [{'aspect': 'flavors', 'sentiment': 'positive'}, {'aspect': 'Thai cuisine', 'sentiment': 'positive'}]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>